# VATEX - pozyskanie części testowej

Liczy i kontroluje to, co zostawiły po sobie skrypty pozyskania z `scripts/prepare_data/`: ile klipów udało się pobrać, jakiej są jakości i czy materiał nadaje się do eksperymentów. Sam niczego nie pobiera. Korzysta wyłącznie z biblioteki standardowej, więc działa niezależnie od stanu środowiska z PyTorch.

**Wymaga:** `scripts/prepare_data/vatex_download.py --split validation` (pobranie klipów i `vatex_report_test.csv`), a przy sekcji C5 także `scripts/prepare_data/vatex_leak_filter.py --split validation` (`vatex_split_test.csv`). Procedura uruchomienia obu jest opisana w dwóch sekcjach niżej.

**Zapisuje:**

| plik | co zawiera | gdzie powstaje |
|---|---|---|
| `data/interim/vatex/work/clips_retried_test.txt` | identyfikatory klipów zgłoszonych do ponowienia | blok B1 |
| `data/interim/vatex/work/clips_too_short_test.csv` | klipy krótsze niż próg, do przeglądu ręcznego | blok C4 |
| `results/figures/vatex_resolution.png` | rozkład rozdzielczości pobranego materiału | blok C1a |

**Dalej:** `vatex_03_test_annotations.ipynb` robi z tych plików znaczniki zapytań i plik zapytań w `data/annotations/vatex/`.

## Skrypty, które trzeba uruchomić wcześniej

### `scripts/prepare_data/vatex_download.py` - pobiera

Czyta `vatex_validation_v1.0.json`, dla każdego z 3000 rekordów wycina z YouTube dziesięciosekundowy fragment i zapisuje go jako `data/raw/vatex/<videoID>.mp4`. Po każdym klipie dopisuje wiersz do `vatex_report_test.csv`: status, rozdzielczość, fps, treść ewentualnego błędu.

Tryby: `--mode descriptions` (eksport zapytań do CSV), `--mode download` (właściwa praca), `--mode check` (inwentaryzacja dostępności bez pobierania). Wznawia pracę, pomijając każdy `videoID` obecny już w raporcie, więc można go przerywać `Ctrl+C` bez straty. Zatrzymuje się sam po 10 blokadach pod rząd.

### `scripts/prepare_data/vatex_retry_errors.py` - przygotowuje ponowienie

Nie pobiera niczego. Skoro skrypt pobierający pomija wszystko, co jest już w raporcie, to ponowienie klipów zakończonych błędem polega na usunięciu ich wierszy. Domyślnie tylko pokazuje, co by zrobił; przy `--execute` zakłada kopię zapasową z datą i dopiero wtedy zapisuje.

Ponawia statusy przejściowe (`error`, `download_error`, `bot`, `rate_limited`, `no_format`, `login_required`), pomija trwałe (`missing`, `private`, `geo_blocked`, `members_only`). Flaga `--check-files` wyłapuje dodatkowo wiersze `ok` bez pliku lub z plikiem poniżej 10 kB i przenosi uszkodzone do `clips/rejected/`; bez tego przeniesienia ponowienie nic by nie dało, bo skrypt pobierający pomija klip, gdy plik docelowy istnieje.

## Procedura krok po kroku

Wszystko poniżej uruchamia się z wiersza poleceń, z korzenia repozytorium. Skrypty liczą ścieżki od korzenia repozytorium (`Path(__file__)`), więc katalog roboczy nie wpływa na to, gdzie zapisują pliki; ścieżki względne podawane w argumentach rozwiązują się natomiast względem katalogu, z którego je uruchamiasz.

| # | Co | Gdzie | Kontrola |
|---|---|---|---|
| 1 | `python scripts/prepare_data/vatex_candidates.py --split validation` | wiersz poleceń | powstaje `vatex_test_candidates.txt`, 3000 identyfikatorów |
| 2 | `python scripts/prepare_data/vatex_download.py --split validation --mode download --limit 20` | wiersz poleceń | sekcja A, czy statusy i rozdzielczości wyglądają zdrowo |
| 3 | `python scripts/prepare_data/vatex_download.py --split validation --mode download` | wiersz poleceń, ok. 13 h | sekcja A po zakończeniu |
| 4 | blok B1, zapis listy ponawianych | notatnik | powstaje `work/clips_retried_test.txt` |
| 5 | `python scripts/prepare_data/vatex_retry_errors.py --split validation --execute --include-age-restricted --check-files` | wiersz poleceń | podgląd bez `--execute` |
| 6 | `python scripts/prepare_data/vatex_download.py --split validation --mode download --cookies-from-browser firefox --no-force-keyframes --pause 3` | wiersz poleceń | - |
| 7 | blok B2, porównanie długości | notatnik | decyzja o odzyskanych klipach |
| 8 | sekcje C i D | notatnik | podsumowanie materiału |

**Krok 1 jest tu inwentarzem, nie wyborem.** Część testowa to cały podzbiór walidacyjny VATEX, więc lista wypisuje wszystkie 3000 klipów i niczego nie rozstrzyga. Powstała ona po pobraniu tych klipów, więc, inaczej niż dla części deweloperskiej, nie dowodzi, że skład nie zależał od dostępności. Nie musi: składem było "wszystkie", a to nie jest wybór, na który dostępność mogła wpłynąć. Lista istnieje po to, żeby obie części wchodziły do pobierania tą samą drogą.

Eksport opisów nie jest osobnym krokiem: `--mode download` dorabia `work/vatex_descriptions_test.csv` na starcie, gdy pliku nie ma. Do przegenerowania go osobno służy `--mode descriptions`, który czyta tylko plik jsona i działa lokalnie, w sekundy.

Podgląd postępu w trakcie długiego przebiegu, w drugim oknie PowerShella:

```powershell
Get-Content data/interim/vatex/vatex_report_test.csv -Wait -Tail 20
```

**Uwaga do kroku 5.** `--cookies-from-browser` wymaga konta zapasowego (nie głównego, wiki yt-dlp ostrzega przed banem) i zamkniętego Firefoksa, bo YouTube rotuje ciasteczka w otwartych kartach. `--no-force-keyframes` ratuje klipy padające na awarii ffmpeg, ale kosztem precyzji cięcia, dlatego kontrole B1 i B2 są obowiązkowe.

## A. Stan pobierania

### A1 - przegląd ogólny

Ile klipów przetworzono i ile z nich się pobrało.

Blok startowy: definiuje ścieżki i funkcje pomocnicze używane przez wszystkie pozostałe, więc po restarcie jądra uruchamiany jest jako pierwszy. Poza tym jest samodzielny i można go powtarzać w trakcie pobierania.

In [ ]:

import csv
import re
import statistics
import subprocess
import sys
from collections import Counter
from pathlib import Path

ROOT     = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# the clip-duration band is shared with the development part: settings is
# stdlib-only, so importing it keeps this notebook independent of PyTorch
from src.utils import settings

DATA_DIR = ROOT / "data" / "interim" / "vatex"
WORK_DIR = DATA_DIR / "work"   # intermediate files of a single procedure
CLIPS    = ROOT / "data" / "raw" / "vatex"
SCRIPTS  = ROOT / "scripts"
REPORT   = DATA_DIR / "vatex_report_test.csv"
NOMINAL = 3000
COL = {}   # report header: column name -> index

# status vocabulary of the report, shared by several cells below
PERMANENT = {"missing", "private", "geo_blocked", "members_only", "corrupt"}
RETRIED   = {"error", "download_error", "bot", "rate_limited",
             "no_format", "login_required", "age_restricted"}
RETRY_LIST = WORK_DIR / "clips_retried_test.txt"

def load():
    """Fresh read of the report - the file grows during the download."""
    if not REPORT.exists():
        raise FileNotFoundError(f"Missing {REPORT}. First steps 1-2 of the procedure.")
    with open(REPORT, encoding="utf-8-sig", newline="") as f:
        rows = [r for r in csv.reader(f, delimiter=";") if r]
    if not rows:
        return []
    if rows[0][0] != "videoID":
        raise ValueError("Report without a header - reading by column names is impossible.")
    COL.clear()
    COL.update({name: i for i, name in enumerate(rows[0])})
    return rows[1:]

def field(r, name):
    """Column value by name - the report has two height and two fps columns."""
    i = COL.get(name)
    if i is None:
        raise KeyError(f"Missing column '{name}' in the report. Columns: {list(COL)}")
    return r[i].strip() if i < len(r) else ""

def numbers(rows, column, convert=float, status="ok"):
    """Numeric values from one column + the number of unparsable rows."""
    values, bad = [], 0
    for r in rows:
        if status and field(r, "status") != status:
            continue
        t = field(r, column).replace(",", ".")
        if not t:
            bad += 1
            continue
        try:
            values.append(convert(float(t)))
        except ValueError:
            bad += 1
    return values, bad

def pl(x, places=1):
    """Number with a decimal comma."""
    return f"{x:.{places}f}".replace(".", ",")

def file_duration(path):
    """Duration from the container metadata (no picture decoding)."""
    w = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "csv=p=0", str(path)],
                       check=False, capture_output=True, text=True, timeout=30)
    try:
        return float(w.stdout.strip())
    except ValueError:
        return None

def ready_files():
    """The .mp4 files without the temporary yt-dlp fragments (X.f299.mp4)."""
    return [p for p in CLIPS.glob("*.mp4") if not re.search(r"\.f\d+$", p.stem)]

In [ ]:
data = load()
statuses = Counter(field(r, "status") for r in data)
n, ok = len(data), statuses.get("ok", 0)

print(f"data directory: {DATA_DIR}")
if n == 0:
    print("the report is empty - the download has not started yet")
else:
    print(f"processed      {n:>6} of {NOMINAL}   ({pl(100*n/NOMINAL)}%)")
    print(f"downloaded     {ok:>6}             ({pl(100*ok/n)}%)")
    print(f"NOT downloaded {n-ok:>6}             ({pl(100*(n-ok)/n)}%)")
    print(f"remaining      {NOMINAL-n:>6}")

### A2 - podział wg przyczyny

Statusy trwałe (`missing`, `private`, `geo_blocked`, `members_only`, `corrupt`) to ubytek zbioru; pozostałe to problem techniczny przebiegu i nadają się do ponowienia. Jeden status potrafi ukrywać kilka różnych awarii, więc wypisywane są też najczęstsze komunikaty.

In [ ]:
data = load()
statuses = Counter(field(r, "status") for r in data)
n, ok = len(data), statuses.get("ok", 0)

permanent = sum(v for s, v in statuses.items() if s in PERMANENT)
to_retry = n - ok - permanent

for s, count in statuses.most_common():
    mark = "" if s == "ok" else ("[permanent]" if s in PERMANENT else "[retry]")
    print(f"{s:<20} {count:>6}  {pl(100*count/n):>5}%  {mark}")

print(f"\npermanent losses {permanent:>6}  {pl(100*permanent/n):>5}%")
print(f"to retry         {to_retry:>6}  {pl(100*to_retry/n):>5}%")

def pattern(t):
    t = re.sub(r"\[youtube\]\s*\S+:", "[youtube] <ID>:", t)
    t = re.sub(r"https?://\S+", "<URL>", t)
    return re.sub(r"\d{3,}", "<N>", t).strip()

for status in sorted(RETRIED & set(statuses)):
    msgs = Counter(pattern(field(r, "message"))
                   for r in data if field(r, "status") == status)
    print(f"\n=== {status} ({sum(msgs.values())}) ===")
    for text, count in msgs.most_common(4):
        print(f"{count:>5}  {text[:110]}")

if to_retry > 0.02 * n:
    print("\n>>> Section B: retry before computing the summary.")

## B. Ponawianie: dwie kontrole wokół `vatex_retry_errors.py`

Ponawianie z flagą `--no-force-keyframes` jest kompromisem: ratuje klipy, które padają na awarii ffmpeg przy przewijaniu w głąb filmu, ale bez wymuszonej klatki kluczowej ffmpeg tnie do najbliższej istniejącej, więc okno może się przesunąć względem adnotacji VATEX.

Przesunięcie uderza przede wszystkim w eksperyment z lokalizacją czasową akcji, gdzie mierzone jest właśnie trafianie we właściwy moment. Byłby to błąd systematyczny, a więc taki, który nie wygląda jak szum, tylko jak wynik. Dlatego kompromis wolno przyjąć tylko wtedy, gdy da się go zmierzyć.

Między B1 a B2 wykonywane są kroki 4 i 5 z procedury.

### B1 - lista identyfikatorów przed ponowieniem

**Zapisuje:** `data/interim/vatex/work/clips_retried_test.txt` - identyfikatory klipów o statusie przejściowym, jeden na wiersz. Blok ostrzega, gdy plik istnieje, bo nadpisanie kasuje listę z poprzedniego ponowienia, a bez niej B2 nie odróżni klipów ciętych precyzyjnie od reszty.

In [ ]:
data = load()
retry_ids = [field(r, "videoID") for r in data if field(r, "status") in RETRIED]

if RETRY_LIST.exists():
    print(f"WARNING: {RETRY_LIST.name} already exists ({len(RETRY_LIST.read_text(encoding='utf-8').split())} IDs).")
    print("Overwriting erases the list from the previous retry - if B2 has not")
    print("used it yet, rename the old file first.")

WORK_DIR.mkdir(parents=True, exist_ok=True)
RETRY_LIST.write_text("\n".join(retry_ids), encoding="utf-8")
print(f"\nsaved {len(retry_ids)} IDs to {RETRY_LIST}")
print("Breakdown:", dict(Counter(field(r, "status") for r in data
                                 if field(r, "status") in RETRIED).most_common()))
print("\nNow steps 4 and 5 of the procedure, then come back to B2.")

### B2 - kontrola cięcia po ponowieniu

Porównanie długości odzyskanych klipów z resztą zbioru. Mediana odzyskanych przy 10 s i udział poza tolerancją podobny do reszty oznacza, że kompromis nic nie kosztował i klipy zostają. Odchylenie systematyczne oznacza, że trzeba je usunąć ze zbioru i odnotować to jako świadomą decyzję, a nie przypadkowy ubytek.

In [ ]:
TOL = 0.5

ids = RETRY_LIST.read_text(encoding="utf-8").split() if RETRY_LIST.exists() else []
if not ids:
    print("Missing clips_retried_test.txt - run B1 first.")
else:
    # "is not None", not "if d": a zero-length clip is an outlier, not a gap
    recovered = [d for d in (file_duration(CLIPS / f"{v}.mp4") for v in ids
                             if (CLIPS / f"{v}.mp4").exists()) if d is not None]
    id_set = set(ids)
    rest_files = [p for p in ready_files() if p.stem not in id_set]
    rest = [d for d in (file_duration(p) for p in sorted(rest_files)[::20])
            if d is not None]

    print(f"of {len(ids)} IDs on the list, files recovered: {len(recovered)}\n")
    for name, durs in [("recovered", recovered), ("rest", rest)]:
        if not durs:
            print(f"{name:<10} no files to measure")
            continue
        outside = sum(1 for d in durs if abs(d - 10.0) > TOL)
        print(f"{name:<10} n={len(durs):>4}  median={pl(statistics.median(durs),2)} s  "
              f"range {pl(min(durs),2)}-{pl(max(durs),2)} s  "
              f"outside tolerance {outside} ({pl(100*outside/len(durs))}%)")

    if recovered and rest:
        diff = abs(statistics.median(recovered) - statistics.median(rest))
        print(f"\ndifference of medians: {pl(diff,2)} s")
        print("OK - the clips stay." if diff <= TOL else
              ">>> The recovered clips deviate. Consider removing them from the dataset.")

## C. Pomiary materiału

### C1 - rozdzielczość

Kubełki to przedziały wysokości, nie klasy "480p": ponad 10% klipów ma wysokość niestandardową (128, 320, 352, 568, 640, 854 px), więc etykieta "480p" obejmowałaby też klip 470 px. Granice domknięte od góry: przedział `(dolna, górna]`. Agregaty liczone są tym samym predykatem co kubełki - wcześniejsza wersja brała "<= 480p" jako dopełnienie zbioru `h >= 720` i zawyżała udział materiału słabej jakości.

In [ ]:
data = load()

BOUNDS          = [144, 240, 360, 480, 720, 1080]   # upper ends of the buckets
SMALL_THRESHOLD = 480
STANDARD        = set(BOUNDS)

heights, without_height = numbers(data, "file_height", int)

if not heights:
    print("no resolution data")
else:
    m = len(heights)
    ok_rows = sum(1 for r in data if field(r, "status") == "ok")

    labels, counts = [], []
    lower = 0
    for upper in BOUNDS:
        labels.append(f"<={upper}" if lower == 0 else f"{lower+1}-{upper}")
        counts.append(sum(1 for h in heights if lower < h <= upper))
        lower = upper
    labels.append(f">{BOUNDS[-1]}")
    counts.append(sum(1 for h in heights if h > BOUNDS[-1]))

    assert sum(counts) == m, "the buckets do not add up to the total"

    print(f"clips with a measured height: {m}   ('ok' rows: {ok_rows})")
    if without_height:
        print(f"WARNING: {without_height} 'ok' rows without a height - check section C2")
    print()
    for label, count in zip(labels, counts):
        print(f"{label:<10} {count:>6}  {pl(100*count/m):>5}%")

    small = sum(1 for h in heights if h <= SMALL_THRESHOLD)
    hd    = sum(1 for h in heights if h >= 720)
    print(f"\n<= {SMALL_THRESHOLD} px    {small:>6}  {pl(100*small/m):>5}%")
    print(f">  {SMALL_THRESHOLD} px    {m-small:>6}  {pl(100*(m-small)/m):>5}%")
    print(f">= 720 px    {hd:>6}  {pl(100*hd/m):>5}%")
    print(f"median: {int(statistics.median(heights))} px")

    nonstd = [h for h in heights if h not in STANDARD]
    print(f"\nnon-standard height: {len(nonstd)}  ({pl(100*len(nonstd)/m)}%)")
    if nonstd:
        frequent = Counter(nonstd).most_common(6)
        print("  most frequent:", ", ".join(f"{h} px x{c}" for h, c in frequent))

### C1a - rysunek rozkładu rozdzielczości

**Zapisuje:** `results/figures/vatex_resolution.png`. Gdy `THESIS_FIGURES` wskazuje katalog rysunków pracy, powstaje tam kopia pod nazwą `rozdzielczosc_vatex.png`.

Blok korzysta z kubełków policzonych w C1 i niczego nie liczy drugi raz. Import matplotliba i modułu stylu jest lokalny i osłonięty: na maszynie bez tych bibliotek blok wypisuje komunikat i kończy się bez błędu, a reszta notatnika działa dalej (wyjątek zapisany w `CLAUDE.md`, w akapicie o notatnikach na samej bibliotece standardowej).

In [ ]:
# The figure directory of the thesis repository, when there is one. A copy lands
# there under the name the .tex sources use (Polish), while the file in
# results/figures/ keeps the name of this repository (English).
THESIS_FIGURES = None        # e.g. Path(r"C:\Users\PC\...\praca magisterska\rysunki")

try:
    import matplotlib.pyplot as plt

    from src.evaluation import figures
except ImportError as error:
    # This notebook runs on a machine without the ML stack on purpose; a missing
    # plotting library skips the figure instead of breaking the procedure.
    print(f"no plotting stack ({error}) - figure skipped, the rest of the "
          f"notebook is unaffected")
else:
    if "counts" not in dir() or "labels" not in dir():
        print("run section C1 first - this cell draws what C1 counted")
    else:
        figures.style()
        PCT = figures.percent_sign()

        # read-only copies: the cell has to survive being run twice without C1
        bucket_names = list(labels)
        bucket_counts = list(counts)
        total = sum(bucket_counts)
        shares = [100 * c / total for c in bucket_counts]

        def axis_label(text):
            """Bucket label with the comparison signs LaTeX can typeset."""
            if text.startswith("<="):
                return r"$\leq$" + text[2:]
            if text.startswith(">"):
                return r"$>$" + text[1:]
            return text

        fig, axis = plt.subplots(figsize=(6.4, 4.0))
        positions = range(len(bucket_names))
        axis.bar(positions, shares, 0.74, color=figures.OKABE_ITO[0],
                 edgecolor="white", linewidth=0.6)

        # the threshold of "poor quality" sits on a bucket boundary, so the line
        # goes between two bars rather than through one
        if SMALL_THRESHOLD in BOUNDS:
            boundary = BOUNDS.index(SMALL_THRESHOLD)
            small_share = 100 * sum(bucket_counts[:boundary + 1]) / total
            axis.axvline(boundary + 0.5, color="0.35", ls="--", lw=1.2)
            axis.text(boundary + 0.55, axis.get_ylim()[1] * 0.95,
                      f"do {SMALL_THRESHOLD} px: {small_share:.1f} {PCT}".replace(".", ","),
                      rotation=90, va="top", ha="left", fontsize=8.5, color="0.35")

        axis.set_xticks(list(positions))
        axis.set_xticklabels([axis_label(n) for n in bucket_names],
                             rotation=45, ha="right", fontsize=9)
        axis.set_xlabel("wysokość klatki [px]")
        axis.set_ylabel(f"udział klipów [{PCT}]")
        fig.tight_layout()

        figures.save(fig, "vatex_resolution", THESIS_FIGURES, "rozdzielczosc_vatex")


### C2 - pliki na dysku a raport

Zgodność liczby plików `.mp4` ze statusami `ok`, rozmiar zbioru i wykrycie plików uszkodzonych albo osieroconych.

In [ ]:
data = load()
CORRUPT_THRESHOLD = 10 * 1024

all_files = list(CLIPS.glob("*.mp4"))
fragments = [p for p in all_files if re.search(r"\.f\d+$", p.stem)]
files = {p.stem: p.stat().st_size for p in ready_files()}

ok_ids = {field(r, "videoID") for r in data if field(r, "status") == "ok"}
missing_file = ok_ids - set(files)
corrupt = [v for v in ok_ids & set(files) if files[v] < CORRUPT_THRESHOLD]
orphaned = sorted(set(files) - ok_ids)
size = sum(files.values())

print(f"ready .mp4 files    {len(files):>6}")
print(f"'ok' statuses       {len(ok_ids):>6}")
print(f"size                {pl(size/1024**3, 2):>6} GB")
print(f"average per clip    {pl(size/max(len(files),1)/1024**2, 2):>6} MB")
print(f"\nmissing file despite 'ok'  {len(missing_file):>4}   (should be 0)")
print(f"files < {CORRUPT_THRESHOLD//1024} kB              {len(corrupt):>4}   (should be 0)")
print(f"files without an 'ok' row  {len(orphaned):>4}")
print(f"yt-dlp fragments in progress {len(fragments):>4}")

if orphaned:
    print("\nFiles without an 'ok' row - examples:")
    for v in orphaned[:5]:
        print("   ", v)
    print("  Ordinary videoIDs: the download is in progress or a leftover")
    print("  from an earlier run.")

if missing_file or corrupt:
    print(f"\n>>> {SCRIPTS / 'vatex_retry_errors.py'} --execute --check-files")

### C3 - liczba klatek na sekundę

Okna obu modeli ruchu (SlowFast, X-CLIP) liczone są w klatkach źródłowych, więc przy innym fps to samo okno obejmuje inny odcinek czasu; stąd ważniejszy jest rozstęp niż sama mediana.

In [ ]:
data = load()

fps_raw, without_fps = numbers(data, "file_fps")
fps_values = [round(v, 1) for v in fps_raw]

if not fps_values:
    print("no fps data")
else:
    distribution = Counter(fps_values)
    print(f"clips with a measured fps: {len(fps_values)}")
    if without_fps:
        print(f"WARNING: {without_fps} 'ok' rows without fps")
    print(f"distinct values: {len(distribution)}\n")

    print("most frequent:")
    for v, count in distribution.most_common(6):
        print(f"{pl(v):>6} fps  {count:>6}  {pl(100*count/len(fps_values)):>5}%")

    print(f"\nmedian: {pl(statistics.median(fps_values))} fps")
    print(f"range:  {pl(min(fps_values))} - {pl(max(fps_values))} fps")
    others = len(fps_values) - distribution[30.0] - distribution[25.0]
    print(f"other than 30 and 25 fps: {others}  ({pl(100*others/len(fps_values))}%)")

### C4 - długości klipów

Długości pochodzą z kolumny `duration` raportu, zmierzonej przez `vatex_download.py` w trakcie pobierania, więc blok nie uruchamia `ffprobe` i działa natychmiast.

Próg jest jednostronny. Klip dłuższy od nominalnego jest nieszkodliwy - ffmpeg dodał margines przy cięciu do klatki kluczowej, a opisywane zdarzenie nadal jest w środku. Klip krótszy oznacza brakujący materiał: nagranie źródłowe skończyło się, zanim zamknęło się okno adnotacji.

Klipy odrzucone ręcznie mają status `corrupt`, więc wypadają z grupy `ok`; raportowane są osobno, żeby po wykluczeniu dało się odtworzyć stan sprzed decyzji.

**Zapisuje:** `data/interim/vatex/work/clips_too_short_test.csv` - identyfikator i długość klipów poniżej progu, od najkrótszego. Służy do przeglądu ręcznego: obejrzeć te klipy, przenieść odrzucone do `clips/rejected/`, a potem uruchomić `python scripts/prepare_data/vatex_exclude_corrupt.py --execute`.

In [ ]:
NOMINAL_DURATION = settings.VATEX_NOMINAL_S
SHORT_THRESHOLD  = settings.VATEX_MIN_CLIP_S
LONG_THRESHOLD   = settings.VATEX_MAX_CLIP_S

data = load()

if "duration" not in COL:
    print("The report has no 'duration' column.")
    print("Run once:  python scripts\\vatex_fill_durations.py --execute")
else:
    durations = {field(r, "videoID"): float(field(r, "duration").replace(",", "."))
                 for r in data
                 if field(r, "status") == "ok" and field(r, "duration")}
    without_duration = sum(1 for r in data
                           if field(r, "status") == "ok" and not field(r, "duration"))
    rejected = [float(field(r, "duration").replace(",", "."))
                for r in data
                if field(r, "status") == "corrupt" and field(r, "duration")]

    short = {v: d for v, d in durations.items() if d < SHORT_THRESHOLD}
    long  = {v: d for v, d in durations.items() if d > LONG_THRESHOLD}
    m = len(durations)

    print(f"clips with a measured duration: {m}")
    if without_duration:
        print(f"no duration despite status 'ok': {without_duration}  <- check section C2")
    print(f"median: {pl(statistics.median(durations.values()), 2)} s")
    print(f"range:  {pl(min(durations.values()), 2)} - {pl(max(durations.values()), 2)} s")
    print(f"\nshorter than {pl(SHORT_THRESHOLD)} s:  {len(short):>5}  ({pl(100*len(short)/m)}%)")
    print(f"longer than {pl(LONG_THRESHOLD)} s:   {len(long):>5}  "
          f"({pl(100*len(long)/m)}%)   <- harmless")

    short_rejected = sum(1 for d in rejected if d < SHORT_THRESHOLD)
    if rejected:
        before = m + len(rejected)
        print("\nbalance before the manual exclusion:")
        print(f"  acquired clips               {before:>5}")
        print(f"  shorter than {pl(SHORT_THRESHOLD)} s           "
              f"{len(short) + short_rejected:>5}  "
              f"({pl(100*(len(short)+short_rejected)/before)}% of acquired)")
        print(f"  of them rejected manually    {len(rejected):>5}"
              + ("" if short_rejected == len(rejected)
                 else f"  (including {len(rejected)-short_rejected} not because of the duration)"))
        print(f"  remaining in the set         {m:>5}")

    # intervals [lower, upper), not cumulative
    print("\ndistribution of the shorter ones (in intervals):")
    INTERVALS = [(0, 1, "below 1 s"), (1, 3, "1-3 s"), (3, 5, "3-5 s"),
                 (5, 7, "5-7 s"), (7, 9, "7-9 s"), (9, SHORT_THRESHOLD, "9-9,5 s")]
    for lower, upper, label in INTERVALS:
        count = sum(1 for d in short.values() if lower <= d < upper)
        print(f"  {label:<12} {count:>5}  ({pl(100*count/m if m else 0)}%)")
    print(f"  {'total':<12} {len(short):>5}")

    TARGET = WORK_DIR / "clips_too_short_test.csv"
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with open(TARGET, "w", encoding="utf-8", newline="") as f:
        f.write("videoID;duration\n")
        for v, d in sorted(short.items(), key=lambda p: p[1]):
            f.write(f"{v};{d:.2f}\n")          # decimal point: this is a data file
    print(f"\nsaved {len(short)} entries to {TARGET.name}")
    print("\nNext, manually: view these clips and move the rejected ones to clips\\rejected\\,")
    print("and then:  python scripts\\vatex_exclude_corrupt.py --execute")

### C5 - skład zbioru testowego

Klipy VATEX pochodzą ze zbioru walidacyjnego Kinetics-600, do którego trafiła część nagrań ze zbioru treningowego Kinetics-400, a na nim trenowano badane modele ruchu. Dla takiego klipu nie da się rozstrzygnąć, czy model rozpoznał zdarzenie, czy odtworzył je z pamięci, więc wypada ze zbioru testowego.

Odfiltrowanie nie usuwa znajomości samych kategorii: modele widziały inne nagrania z tych klas Kinetics-600, które pokrywa słownik Kinetics-400. To nie jest wada zbioru, tylko przedmiot pomiaru w E2, stąd drugi podział.

Liczby pochodzą z `vatex_split_test.csv`, który tworzy `scripts/prepare_data/vatex_leak_filter.py`; bez uruchomienia tego skryptu blok tylko o tym informuje. Skrypt nie rusza `vatex_report_test.csv`: raport opisuje pozyskanie, a przynależność do zbioru treningowego jest właściwością nagrania, nie problemem z pobraniem. Klipy z przeciekiem zostają na dysku, bo używa ich kontrola poprawności implementacji.

In [ ]:
SPLIT = DATA_DIR / "vatex_split_test.csv"

if not SPLIT.exists():
    print("Missing vatex_split_test.csv.")
    print("Run:  python scripts\\prepare_data\\vatex_leak_filter.py --split validation")
else:
    with open(SPLIT, encoding="utf-8-sig", newline="") as f:
        split = list(csv.DictReader(f, delimiter=";"))

    leak = [r for r in split if r["leak_k400"] == "yes"]
    test = [r for r in split if r["leak_k400"] == "no"]
    in_vocab = [r for r in test if r["class_in_k400_vocab"] == "yes"]
    outside  = [r for r in test if r["class_in_k400_vocab"] == "no"]
    n = len(split)

    # built from the "ok" rows, i.e. already after the manual exclusion
    print(f"after the duration check   {n:>6}")
    print(f"leak K400 train            {len(leak):>6}  "
          f"({pl(100*len(leak)/n):>4}%)")
    print(f"TEST SPLIT                 {len(test):>6}  "
          f"({pl(100*len(test)/n):>4}%)")

    m = len(test)
    print("\nsplit for E2 (relative to the test split):")
    print(f"  class in K400 vocab      {len(in_vocab):>6}  "
          f"({pl(100*len(in_vocab)/m):>4}%)")
    print(f"  class outside vocab      {len(outside):>6}  "
          f"({pl(100*len(outside)/m):>4}%)")
    print(f"  classes represented      {len({r['class_k600'] for r in test}):>6}")

    MIN_GROUP = 50
    for name, group in [("in vocab", in_vocab), ("outside vocab", outside)]:
        if len(group) < MIN_GROUP:
            print(f"\n>>> Group '{name}' has {len(group)} clips, below the threshold {MIN_GROUP}.")

## D. Środowisko

Wersje narzędzi i data. Bez nich liczby nie są odtwarzalne, bo zbiory oparte na YouTube zmieniają się w czasie.

In [ ]:
def version(cmd):
    try:
        w = subprocess.run(cmd, check=False, capture_output=True, text=True, timeout=30,
                           encoding="utf-8", errors="replace")
        return (w.stdout or w.stderr).strip().splitlines()[0]
    except Exception:
        return "NOT FOUND"

print(f"yt-dlp    {version(['yt-dlp', '--version'])}")
print(f"ffprobe   {version(['ffprobe', '-version'])}")
print(f"deno      {version(['deno', '--version'])}")

## Łączny czas trwania klipów

Suma kolumny `duration` dla wierszy `ok`, nie plików na dysku: `ffprobe` po 2560 plikach trwa minuty, a przy przerwanym pobieraniu wzorzec `*.mp4` złapałby też fragmenty yt-dlp.

In [ ]:
data = load()
durations_ok, without_duration = numbers(data, "duration")

total = sum(durations_ok)
print(f"VATEX: {len(durations_ok)} clips with a measured duration"
      + (f"  ({without_duration} without a duration!)" if without_duration else ""))
print(f"total duration: {pl(total, 1)} s = {int(total//3600)}:{int(total%3600//60):02d}:{int(total%60):02d} (hh:mm:ss)")
print(f"average per clip: {pl(total/len(durations_ok), 2)} s")

# consistency check against the disk - should agree with section C2
print(f"\n.mp4 files on disk: {len(ready_files())}")